In [1]:
import pandas as pd

In [2]:
%pwd

'c:\\Customer_churn_project\\notebooks'

In [3]:
import os

os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path

In [5]:

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir:Path
    input_data_path:Path
    transformed_data_path:Path

In [6]:
from src.CustomerChurnPrediction.constants import *
from src.CustomerChurnPrediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:

    def __init__(self, config_filepath=CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self):

        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            input_data_path=config.input_data_path,
            transformed_data_path=config.transformed_data_path
        )

        return data_transformation_config

In [8]:
import pandas as pd
import pandas as pd
from src.CustomerChurnPrediction.utils.logger import logger


In [ ]:

class DataTransformation:
    """Handles cleaning, encoding, and feature transformation for the Telco Churn dataset."""

    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def get_input_data(self) -> pd.DataFrame:
        """Load raw CSV data from the configured path."""
        
        df = pd.read_csv(self.config.input_data_path)
        logger.info(f"Loaded raw data with shape {df.shape}")
        
        return df

    def drop_unnecessary_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop columns that have no predictive value (e.g., customerID)."""
       
        df = df.drop(columns=['customerID'])
        logger.info("Dropped column: customerID")
       
        return df


    def binary_encoding(self, df: pd.DataFrame) -> pd.DataFrame:
        """Encode binary categorical columns (Yes/No, Male/Female) as 1/0."""

        binary_cols = [
            'gender',
            'Partner',
            'Dependents',
            'PhoneService',
            'PaperlessBilling',
            'Churn'
        ]

        df[binary_cols] = df[binary_cols].replace({
            'Yes': 1,
            'No': 0,
            'Male': 1,
            'Female': 0
        })

        logger.info(f"Binary encoded columns: {binary_cols}")
        return df

    def one_hot_encoding(self, df: pd.DataFrame) -> pd.DataFrame:
        """One-hot encode multi-category columns, dropping the first category to avoid redundancy."""

        multi_cat_cols = [
            'MultipleLines',
            'InternetService',
            'OnlineSecurity',
            'OnlineBackup',
            'DeviceProtection',
            'TechSupport',
            'StreamingTV',
            'StreamingMovies',
            'Contract',
            'PaymentMethod'
        ]

        df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
        logger.info(f"One-hot encoded columns: {multi_cat_cols}; new shape {df.shape}")
        
        return df

    def collapse_redundant_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Collapse 'No internet/phone service' dummy columns into single flags to reduce multicollinearity (decided via VIF analysis in notebook)."""

        # Collapse all "No internet service" dummies into one column
        no_internet_cols = [col for col in df.columns if 'No internet service' in col]

        if no_internet_cols:
            df['No_internet_service'] = df[no_internet_cols].any(axis=1).astype(int)
            df = df.drop(columns=no_internet_cols)
            logger.info(f"Collapsed columns into No_internet_service: {no_internet_cols}")

        # Handle PhoneService redundancy similarly
        if 'MultipleLines_No phone service' in df.columns:
            df['No_phone_service'] = df['MultipleLines_No phone service'].astype(int)
            df = df.drop(columns=['MultipleLines_No phone service'])
            logger.info("Collapsed MultipleLines_No phone service into No_phone_service")

        return df

    def bool_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        
        """Convert any boolean columns (created during one-hot encoding) to int (0/1)."""
       
        bool_cols = df.select_dtypes(include='bool').columns

        if len(bool_cols) > 0:
            df[bool_cols] = df[bool_cols].astype(int)
            logger.info(f"Converted bool columns to int: {list(bool_cols)}")

        return df

    def get_transform_data(self) -> pd.DataFrame:
       
        """Run the full transformation pipeline and return the model-ready DataFrame."""

        df = self.get_input_data()

        df = self.drop_unnecessary_columns(df)

        df = self.binary_encoding(df)

        df = self.one_hot_encoding(df)

        df = self.collapse_redundant_columns(df)

        df = self.bool_to_int(df)

        logger.info(f"Transformation pipeline complete. Final shape: {df.shape}")
        
        create_directories([os.path.dirname(self.config.transformed_data_path)])
        df.to_csv(self.config.transformed_data_path,index=False)

In [10]:
config = ConfigurationManager()
data_transformtion_config = config.get_data_transformation_config()
data_transformation = DataTransformation(data_transformtion_config)
data_transformation.get_transform_data()

[2026-06-21 13:59:00,292] 33 CustomerChurnPrediction - INFO - yaml file: config\config.yml loaded successfully
[2026-06-21 13:59:00,296] 50 CustomerChurnPrediction - INFO - created directory at: artifacts
[2026-06-21 13:59:00,302] 50 CustomerChurnPrediction - INFO - created directory at: artifacts/data_transformation
[2026-06-21 13:59:00,352] 11 CustomerChurnPrediction - INFO - Loaded raw data with shape (7032, 21)
[2026-06-21 13:59:00,358] 19 CustomerChurnPrediction - INFO - Dropped column: customerID
[2026-06-21 13:59:00,397] 43 CustomerChurnPrediction - INFO - Binary encoded columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
[2026-06-21 13:59:00,426] 63 CustomerChurnPrediction - INFO - One-hot encoded columns: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']; new shape (7032, 31)
[2026-06-21 13:59:00,437] 76 CustomerChurnPredi

C:\Users\Admin\AppData\Local\Temp\ipykernel_11560\2911164096.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_cols] = df[binary_cols].replace({
